In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [ ]:
import pandas as pd
import numpy as np
import spacy
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from datasets import load_from_disk

In [ ]:
ds = load_from_disk("/content/drive/MyDrive/Projects Preparation/Sentiment Analysis/train")
nlp = spacy.load('en_core_web_sm', disable=['ner'])

In [ ]:
dataframe = ds.to_pandas()

In [ ]:
df = dataframe[:]

In [ ]:
print(df.shape)
df.head()

(25000, 2)


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [ ]:
X, _ = train_test_split(df, train_size=0.4, stratify=df['label'], random_state=42)

In [ ]:
print(X.shape)
X.head()

(10000, 2)


,text,label
13170,It's amazing that actress P.J. Soles didn't be...,1
7914,"The beginning of the 90s brought many ""quirky""...",0
5235,I found out about this film because Jewish Ben...,0
17506,How can anyone argue the fact that Urban Cowbo...,1
5076,Melissa's sixteenth birthday is right around t...,0


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X['text'], X['label'], test_size=0.2, stratify=X['label'], random_state=42)

In [ ]:
print(f"Train size: {X_train.shape}|Test size: {X_test.shape}")

Train size: (8000,)|Test size: (2000,)


#Preprocessing

In [ ]:
def clean_doc(doc):
    tokens = []
    for token in doc:
        if token.dep_ == "neg":
            tokens.append("not")
        elif token.is_alpha and len(token.text) > 2 and token.pos_ not in {"DET", "ADP"}:
            tokens.append(token.lemma_.lower())
    return " ".join(tokens)

##Training sample preprocessing

In [ ]:
X_train = X_train.to_frame()

In [ ]:
docs=[]
docs = list(nlp.pipe(X_train['text']))

In [ ]:
X_train['clean_text'] = [clean_doc(doc) for doc in docs]

In [ ]:
X_train.head()

,text,clean_text
10079,"""Ghost of Dragstrip Hollow"" was one of the man...",ghost dragstrip hollow be one many movie hot r...
21540,This is the best movie I've seen since White a...,this good movie see since white and well roman...
111,There are lots of extremely good-looking peopl...,there be lot extremely good look people movie ...
24776,"I was literally preparing to hate this movie, ...",be literally prepare hate movie believe when s...
3708,How can they from Joe Don Baker (as Bufford Pu...,how can they joe don baker bufford pusser firs...


In [ ]:
X_train.values[:1]

array([['"Ghost of Dragstrip Hollow" was one of the many \'50s movies about hot-rodding teens encountering the supernatural. In this case, the teens can\'t pay the rent for their hangout and get evicted. With nowhere else to go, they decide on an apparently haunted house. As you may have guessed, once they arrive, some weird things start happening. And there\'s a twist at the end.<br /><br />There\'s nothing in this movie that you haven\'t seen in other movies, but it\'s nice entertainment nonetheless. My favorite character was the foul-mouthed parrot. Well, let me rephrase that: he didn\'t talk like a character in a Quentin Tarantino movie, but he said things that we don\'t expect out of a bird. The movie\'s pure hokum, but harmless.',
        'ghost dragstrip hollow be one many movie hot rodde teen encounter supernatural case teen not pay rent their hangout and get evict with nowhere else they decide apparently haunt house you may have guess once they arrive weird thing start happen 

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train['clean_text'])

In [ ]:
X_train_arr = X_train_tfidf.toarray()

In [ ]:
np.unique(y_train)

array([0, 1])

##Test sample preprocessing

In [ ]:
X_test = X_test.to_frame()

In [ ]:
test_docs = list(nlp.pipe(X_test['text']))

In [ ]:
X_test['clean_text'] = [clean_doc(doc) for doc in test_docs]

In [ ]:
X_test_tfidf = tfidf.transform(X_test['clean_text'])

In [ ]:
X_test_arr = X_test_tfidf.toarray()

#Model creation and Training

In [ ]:
model = MultinomialNB()

In [ ]:
history = model.fit(X_train_arr, y_train)

#Model Saving

In [ ]:
joblib.dump(model, "/content/drive/MyDrive/Projects Preparation/Sentiment Analysis/model/sentiment_sm.pkl")

['/content/drive/MyDrive/Projects Preparation/Sentiment Analysis/model/sentiment_sm.pkl']

#Loaading model

In [ ]:
model = joblib.load("/content/drive/MyDrive/Projects Preparation/Sentiment Analysis/model/model.pkl")

#Model Prediction and Evaluation

In [ ]:
y_pre = model.predict(X_test_arr)

In [ ]:
print(classification_report(y_test, y_pre))

              precision    recall  f1-score   support

           0       0.85      0.88      0.87      1000
           1       0.88      0.84      0.86      1000

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000

